In [0]:
from pyspark.sql import functions as F

In [0]:
youtube_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/Volumes/quant_databricks/batch0506/quantcloudrawdatasets/youtube.csv")
)
youtube_df.display()

In [0]:
column_names = youtube_df.columns
new_column_names = []
for old_col_name in column_names:
    new_col_name = old_col_name.replace(" ", "_").lower()
    new_column_names.append(new_col_name)


youtube_df = youtube_df.select(*[F.col(old_col).alias(new_col) for new_col, old_col in zip(new_column_names, youtube_df.columns)])
youtube_df.display()

In [0]:
# small exercise
# I want to extract the best video release year from best_video column

youtube_df = (
    youtube_df.withColumn("best_video_year", F.substring(F.col("best_video"), -4, 4))
)
display(youtube_df)

In [0]:
youtube_df.count()

In [0]:
youtube_df = youtube_df.drop("content_value_index")
display(youtube_df)

In [0]:
youtube_df = youtube_df.distinct()
display(youtube_df)

In [0]:
youtube_df.dropDuplicates(subset=["channel_name", "youtuber_name"]).display()

In [0]:
# Convert df into temp sql table
youtube_df.createOrReplaceTempView("youtube")

In [0]:
%sql
select * from youtube;

In [0]:
%sql
select channel_name, count(channel_name) as count 
from youtube 
group by channel_name
having count(channel_name) > 1;

In [0]:
# Aggregations

df = (
    youtube_df
    .groupBy("channel_name")
    .count()
)
df = df.filter(F.col("count") > 1)
df.display()

In [0]:
df = (
    youtube_df
    .groupBy("channel_name")
    .agg(
        F.count("*").alias("cnt"),
        F.sum("total_subscribers").alias("total_subscribers"),
    )
)
df = df.filter(F.col("cnt") > 1)
df.display()